# Feature Extraction for Fake vs. Real News Classification (1 of 4)

This notebook loads the dataset, preprocesses the text, and extracts linguistic, stylistic, and readability-based identifiers for statistical and machine learning analysis.

### 1. Imports & Setup

In [12]:
# !pip install nltk textblob sklearn readability-lxml better_profanity spacy
# !python -m spacy download en_core_web_sm

In [13]:
# Imports
import pandas as pd
import numpy as np
import nltk
from nltk.corpus import stopwords
from nltk.sentiment import SentimentIntensityAnalyzer
from better_profanity import profanity
from scipy import stats

# NLTK Downloads
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('vader_lexicon')

profanity.load_censor_words()
bad_words_set = {str(w).lower() for w in profanity.CENSOR_WORDSET if w}

stop_words = set(stopwords.words('english'))

sia = SentimentIntensityAnalyzer()



[nltk_data] Downloading package punkt to
[nltk_data]     /Users/ninaelmoyan/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/ninaelmoyan/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /Users/ninaelmoyan/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


### 2. Load Dataset
The dataset contains `Fake.csv` and `True.csv`. We label and combine them later.

In [14]:
df_fake = pd.read_csv("Dataset/Fake.csv")
df_true = pd.read_csv("Dataset/True.csv")

df_fake["label"] = 0
df_true["label"] = 1

df_combined = pd.concat([df_fake, df_true], ignore_index=True)

### 3. Feature Extraction Functions

In [15]:
def word_count(text):
    return len(nltk.word_tokenize(str(text)))

def char_count(text):
    return len(str(text))

def avg_word_length(text):
    words = nltk.word_tokenize(str(text))
    return np.mean([len(w) for w in words]) if words else 0

def lexical_diversity(text):
    words = [w for w in nltk.word_tokenize(str(text)) if w.isalpha()]
    return len(set(words))/len(words) if words else 0

def num_exclamations(text):
    return str(text).count('!')

def num_questions(text):
    return str(text).count('?')

def uppercase_words(text):
    return sum(1 for w in str(text).split() if w.isupper())

def stopword_count(text):
    return sum(1 for w in str(text).split() if w.lower() in stop_words)

def profanity_count(text):
    words = [w.lower() for w in nltk.word_tokenize(str(text))]
    return sum(1 for w in words if w in bad_words_set)

def sentiment_scores(text):
    scores = sia.polarity_scores(str(text))
    return scores['neg'], scores['compound']

### 4. Apply identifiers to data

The following function bundles all identifiers into a single row for each news article.

In [16]:
def extract_text_identifiers(df, column='text'):
    identifiers = pd.DataFrame()
    identifiers['word_count'] = df[column].apply(word_count)
    identifiers['char_count'] = df[column].apply(char_count)
    identifiers['avg_word_length'] = df[column].apply(avg_word_length)
    identifiers['lexical_diversity'] = df[column].apply(lexical_diversity)
    identifiers['num_exclamations'] = df[column].apply(num_exclamations)
    identifiers['num_questions'] = df[column].apply(num_questions)
    identifiers['uppercase_words'] = df[column].apply(uppercase_words)
    identifiers['stopword_count'] = df[column].apply(stopword_count)
    identifiers['profanity_count'] = df[column].apply(profanity_count)
    
    neg, compound = zip(*df[column].apply(sentiment_scores))
    identifiers['negativity'] = neg
    identifiers['compound_sentiment'] = compound
    
    return identifiers

def extract_title_identifiers(df, column='title'):
    identifiers = pd.DataFrame()
    identifiers['word_count'] = df[column].apply(word_count)
    identifiers['char_count'] = df[column].apply(char_count)
    identifiers['avg_word_length'] = df[column].apply(avg_word_length)
    identifiers['lexical_diversity'] = df[column].apply(lexical_diversity)
    identifiers['num_exclamations'] = df[column].apply(num_exclamations)
    identifiers['num_questions'] = df[column].apply(num_questions)
    identifiers['uppercase_words'] = df[column].apply(uppercase_words)
    identifiers['stopword_count'] = df[column].apply(stopword_count)
    identifiers['profanity_count'] = df[column].apply(profanity_count)
    
    neg, compound = zip(*df[column].apply(sentiment_scores))
    identifiers['negativity'] = neg
    identifiers['compound_sentiment'] = compound
    
    return identifiers

### 5. Apply Feature Extraction to Dataset

In [17]:
text_identifiers = extract_text_identifiers(df_combined, column='text')
title_identifiers = extract_title_identifiers(df_combined, column='title')

# Add labels
text_identifiers['label'] = df_combined['label']
title_identifiers['label'] = df_combined['label']

In [18]:
def t_test_identifiers(identifiers_df):
    results = {}
    for col in identifiers_df.columns:
        if col == 'label':
            continue
        fake_vals = identifiers_df[identifiers_df['label']==0][col]
        true_vals = identifiers_df[identifiers_df['label']==1][col]
        t_stat, p_val = stats.ttest_ind(fake_vals, true_vals, equal_var=False)
        results[col] = {'t_stat': t_stat, 'p_val': p_val}
    return pd.DataFrame(results).T

In [19]:
text_ttest_results = t_test_identifiers(text_identifiers)
title_ttest_results = t_test_identifiers(title_identifiers)

In [20]:
print("Text Identifiers T-Test Results:")
display(text_ttest_results)

print("\nTitle Identifiers T-Test Results:")
display(title_ttest_results)

Text Identifiers T-Test Results:


,t_stat,p_val
word_count,9.492750,2.364994e-21
char_count,8.147394,3.822292e-16
avg_word_length,-30.192215,6.357574e-197
lexical_diversity,-18.616062,4.839352e-77
num_exclamations,51.454646,0.000000e+00
num_questions,71.223223,0.000000e+00
uppercase_words,22.949274,9.955497e-116
stopword_count,24.950224,2.327927e-136
profanity_count,45.332147,0.000000e+00
negativity,34.135363,3.817547e-252



Title Identifiers T-Test Results:


,t_stat,p_val
word_count,180.480894,0.000000e+00
char_count,156.956784,0.000000e+00
avg_word_length,-87.761981,0.000000e+00
lexical_diversity,-34.375449,1.631749e-255
num_exclamations,58.162125,0.000000e+00
num_questions,35.405460,1.196513e-268
uppercase_words,144.900184,0.000000e+00
stopword_count,122.617629,0.000000e+00
profanity_count,31.521016,8.699609e-215
negativity,26.321446,1.604540e-151
